### Step 1: Import Libraries & API Keys

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)

sk-proj-


### Step 2: Set up Pushover

In [6]:
# Step 2a -> Setu up account in your browser
# Step 2b -> Set up the app on your phone
# Step 2c -> In the browser create an "Application/API Token"
# Step 2d -> Add the User Key and API token to your .env file
# Step 2e -> Install the Pushover app on your phone and log in with the same account
# Step 2f -> Run the code below to send a test notification to your phone

load_dotenv()

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


### Step 3: Test Pushover

In [7]:
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [8]:
send_notification("Hello from the AI Engineering course!")

### Step 4: Describe Pushover as an LLM tool

In [9]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a notification to the user's phone using the Pushover service.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
            }
        },
        "required": ["message"]
    }
}

### Step 5: Add Pushover to the list of tools for the LLM

In [10]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 6: Calling the tool from an LLM

In [19]:
client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Send a notification to my phone that says telling me what amazing progress I'm making on the AI Engineering course!"}
    ],
    tools=tools,
    tool_choice="auto"
)

# Check if model wants to call a tool
message = response.choices[0].message

In [20]:
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_6aDirdrkqkricQMTiUhM9MKx', function=Function(arguments='{"message":"You\'re making amazing progress on the AI Engineering course! Keep up the great work!"}', name='send_notification'), type='function')])


In [21]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    import json
    args = json.loads(tool_call.function.arguments)

    # Actually send the notification
    send_notification(args["message"])
    print(f"Sent notification with message: {args['message']}")
else:
    print(message.content)

Sent notification with message: You're making amazing progress on the AI Engineering course! Keep up the great work!
